In [ ]:
# Import Libraries and modules

# libraries that are used for analysis and visualization

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Impoting data preprocessing libraries
from sklearn.preprocessing import StandardScaler, MinMaxScaler
## Handling target class imbalance
from imblearn.over_sampling import SMOTE
# from imblearn.combine import SMOTETomek

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Importing model selection libraries.
from sklearn.model_selection import train_test_split


In [ ]:
# Importing metrics for model evaluation.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score,roc_curve, roc_auc_score

In [ ]:
# Importing machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
df=pd.read_csv('../data/cardiovascular_risk_data.csv',na_values=["NA", "N/A", "null", "None", "?", "-"])
df.drop(['id', 'is_smoking', 'diaBP', 'prevalentHyp'], axis=1, inplace=True, errors='ignore')
nan_columns = ['education', 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate']
df.dropna(subset=nan_columns, inplace=True)

In [ ]:
X =df.drop(columns='TenYearCHD')     # independent features
y =df['TenYearCHD']                  # dependent features
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

In [ ]:
def get_feature_types(df):
    numeric    = [c for c in df.columns if df[c].nunique() > 10]
    categorical = [c for c in df.columns if df[c].nunique() <= 10]
    return numeric, categorical

from sklearn.impute import KNNImputer
def impute_glucose(X):
    X = X.copy()
    knn = KNNImputer(weights='distance')
    X['glucose'] = knn.fit_transform(X[['glucose']])
    return X

def clip_outliers(X):
    X = X.copy()
    numeric_features, _ = get_feature_types(X)
    for col in numeric_features:
        q1 = X[col].quantile(0.25)
        q3 = X[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        X[col] = X[col].clip(lower_bound, upper_bound)
    return X

def encode_education(X):
    X = X.copy()
    dummies = pd.get_dummies(X['education'])
    dummies.columns = ['education_1', 'education_2', 'education_3', 'education_4']
    dummies = dummies.astype(int) 
    X = pd.concat([X, dummies], axis=1)
    X = X.drop(['education', 'education_4'], axis=1, errors='ignore')
    return X

def encode_sex(X):
    X = X.copy()
    X['sex'] = X['sex'].map({'M': 1, 'F': 0}) 
    return X

In [ ]:
pipeline = Pipeline(steps=[
    ('impute_glucose',  FunctionTransformer(impute_glucose)),
    ('clip_outliers',   FunctionTransformer(clip_outliers)),
    ('encode_education',FunctionTransformer(encode_education)),
    ('encode_sex',       FunctionTransformer(encode_sex)),
    ('scale', StandardScaler()),
])

In [ ]:
X_train = pipeline.fit_transform(X_train)
X_test  = pipeline.transform(X_test)

## Handling target class imbalance using SMOTE

In [ ]:
print(f'Before Handling Imbalanced class {y_train.value_counts()}')

# Resampling the minority class
smote = SMOTE(random_state=42, sampling_strategy=0.3)
# fit predictor and target variable
X_train, y_train = smote.fit_resample(X_train, y_train)


In [ ]:
print(f'After Handling Imbalanced class {y_train.value_counts()}')

In [ ]:
# empty list for appending performance metric score 
model_result = []
from sklearn.metrics import average_precision_score

In [ ]:
def predict(ml_model_object, model_name,threshold=0.539):
  
  '''
  Pass the model and predict value. 
  Function will calculate all the eveluation metrics and appending those metrics score on model_result table.
  Plotting confusion_matrix for test data.
  ''' 
  
  # model fitting
  ml_model_object.fit(X_train, y_train,)
  
  # predicting value and probability
  y_train_pred = ml_model_object.predict(X_train)
  y_test_pred = ml_model_object.predict(X_test)
  y_train_prob = ml_model_object.predict_proba(X_train)[:,1]
  y_test_prob = ml_model_object.predict_proba(X_test)[:,1]

  y_test_pred = (y_test_prob >= threshold).astype(int)
  
  print(model_name)
  print("\n")
  ''' Performance Metrics ''' 
  # accuracy score  ---->  (TP+TN)/(TP+FP+TN+FN)
  train_accuracy = accuracy_score(y_train, y_train_pred) 
  test_accuracy = accuracy_score(y_test, y_test_pred)
  print(f'train accuracy : {round(train_accuracy,3)}')
  print(f'test accuracy : {round(test_accuracy,3)}')

  # precision score  ---->  TP/(TP+FP)
  train_precision = precision_score(y_train, y_train_pred)
  test_precision = precision_score(y_test, y_test_pred)
  print(f'train precision : {round(train_precision,3)}')
  print(f'test precision : {round(test_precision,3)}')

  avg_precison_train = average_precision_score(y_train, y_train_prob)
  avg_precison_test = average_precision_score(y_test, y_test_prob)
  print(f'avg_precison_train recall : {round(avg_precison_train,3)}')
  print(f'train avg_precison_test : {round(avg_precison_test,3)}')
  

  # recall score  ---->  TP/(TP+FN)
  train_recall = recall_score(y_train, y_train_pred)
  test_recall = recall_score(y_test, y_test_pred)
  print(f'train recall : {round(train_recall,3)}')
  print(f'test recall : {round(test_recall,3)}')
  
  # f1 score  ---->  Harmonic Mean of Precision and Recall
  train_f1 = f1_score(y_train, y_train_pred)
  test_f1 = f1_score(y_test, y_test_pred)
  print(f'train f1 : {round(train_f1,3)}')
  print(f'test f1 : {round(test_f1,3)}')
  print('-'*80)

  # roc_auc score  ---->  It shows how well the model can differentiate between classes.
  train_roc_auc = roc_auc_score(y_train, y_train_prob)
  test_roc_auc = roc_auc_score(y_test, y_test_prob)
  print(f'train roc_auc : {round(train_roc_auc,3)}')
  print(f'test roc_auc : {round(test_roc_auc,3)}')
  print('-'*80)


  ''' plotting Confusion Matrix '''
  ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred)
  plt.title('confusion matrix on Test data', weight='bold')
  plt.show()
  print('-'*80)

  ''' plotting ROC curve '''
  fpr, tpr, threshold = roc_curve(y_test, y_test_prob)
  plt.plot(fpr,tpr, label=f'ROC - {model_name}')
  plt.plot([0,1], [0,1], '--')
  plt.title('ROC curve on Test data', weight='bold')
  plt.xlabel('False Positive Rate----->')
  plt.ylabel('True Positive Rate----->')
  plt.legend(loc=4)


  ''' actual value vs predicted value on test data'''
  d = {'y_actual':y_test, 'y_predict':y_test_pred}
  print(pd.DataFrame(data=d).head(10).T)                   # constructing a dataframe with both actual and predicted values
  print('-'*80)


  # using the score from the performance metrics to create the final model_result.
  model_result.append({'model':model_name,
                       'train_roc_auc':train_roc_auc,
                       'test_roc_auc':test_roc_auc,
                       'train_accuracy':train_accuracy, 
                       'test_accuracy':test_accuracy, 
                       'train_precision':train_precision,
                       'test_precision':test_precision,
                       'train_avg_precison':avg_precison_train,
                       'test_avg_precison':avg_precison_test,
                       'train_recall':train_recall,
                       'test_recall':test_recall,
                       'train_f1':train_f1,
                       'test_f1':test_f1})
  
  print()

In [ ]:
# Checking the optimum value of the k:
accuracy=[]

# Iterating for the optimum value of k
for i in range(1,15):
  knn=KNeighborsClassifier(n_neighbors=i)
  knn.fit(X_train,y_train)
  accuracy.append(knn.score(X_test, y_test))

### plotting the k-value vs accuracy

In [ ]:
plt.title('k-NN Varying number of neighbors')
plt.plot(range(1,15), accuracy)
plt.xticks(range(1,15))
plt.xlabel('number of neighbours')
plt.ylabel('Accuracy')
plt.show()  

### Insight: The best accuracy is at K=14. For binary classification, k is typically an odd number (to prevent ties) of at least three. Hence use k=13

In [ ]:
rf_params = {'n_estimators': [200, 300, 500],     #number of trees 
             'max_features': ["log2", "sqrt"],  # maximum number of features considered when splitting a node
             'max_depth': [5,10,15,20],        # maximum number of levels allowed
             'min_samples_split': [10, 20, 30],     # minimum number of samples necessary in a node to cause node splitting
             'min_samples_leaf': [5, 10, 20]}

### performing Hyperparameter Tunning using RandomizedSearchCV

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_randomsearch = RandomizedSearchCV(estimator=rf, param_distributions=rf_params, n_iter=15, cv=5, verbose=2, n_jobs=-1)

rf_randomsearch.fit(X_train,y_train)

optimal_model = rf_randomsearch.best_estimator_
optimal_model

In [ ]:
predict(LogisticRegression(class_weight='balanced', C=1, max_iter=1000), 'LogisticRegression',0.5165)
predict(SVC(probability=True), 'SVM')
predict(DecisionTreeClassifier(), 'DecisionTree')
predict(KNeighborsClassifier(n_neighbors=1), 'KNN')
predict(optimal_model, 'RandomForest')

In [ ]:
#converting model_result to DataFrame
model_result = pd.DataFrame(model_result)
round(model_result,3)

In [ ]:
# plotting graph to compaire model performance of all the models
fig, axs = plt.subplots(1,2, figsize=(12,4))

sns.barplot(x=model_result['model'], y=model_result['test_recall'], ax=axs[0])   # Model vs Recall score
sns.barplot(x=model_result['model'], y=model_result['test_roc_auc'], ax=axs[1])# Model vs roc_auc score
     
for ax in axs:
    ax.tick_params(axis='x', rotation=90)

plt.tight_layout()